# Elicitation Robustness Check

Tests whether the declarative-evaluative gap persists across meaningfully different
elicitation strategies. If the gap holds across all five template types, it rules out
prompting artifacts as a confounder.

**Design:**
- 3 concepts (closed captions, color contrast, page title)
- 5 template types (cloze, direct_question, instruction, evaluative, scenario)
- 3 models (Pythia 2.8B, Pythia 1B as predicted null, GPT-2-large)
- 100 max tokens, temperature 0, greedy decoding

## Setup

In [ ]:
import json
import torch
import pandas as pd
from transformer_lens import HookedTransformer
from pathlib import Path

In [ ]:
# Check device
if torch.backends.mps.is_available():
    device = 'mps'
elif torch.cuda.is_available():
    device = 'cuda'
else:
    device = 'cpu'
print(f'Using device: {device}')

## Load Prompts

In [ ]:
# Load prompts from JSONL
prompt_file = Path('../data/elicitation_control_prompts.jsonl')

prompts = []
with open(prompt_file) as f:
    for line in f:
        prompts.append(json.loads(line.strip()))

print(f'Loaded {len(prompts)} prompts')
print(f'Concepts: {sorted(set(p["concept"] for p in prompts))}')
print(f'Template types: {sorted(set(p["template_type"] for p in prompts))}')

## Run Models

Load one model at a time to stay within memory. Collect all completions into a
single results list, then build a DataFrame at the end.

In [ ]:
def run_prompts(model, model_name, prompts, max_tokens=100):
    """Run all prompts through a model and return results."""
    results = []
    for p in prompts:
        output = model.generate(
            p['prompt'],
            max_new_tokens=max_tokens,
            temperature=0,
            verbose=False
        )
        completion = output[len(p['prompt']):]
        results.append({
            'model': model_name,
            'concept': p['concept'],
            'template_type': p['template_type'],
            'prompt_id': p['prompt_id'],
            'prompt': p['prompt'],
            'completion': completion,
            'full_output': output
        })
        print(f"  [{p['template_type']:16}] {p['prompt'][:50]:50} → {completion[:80]}")
    return results

### Pythia 2.8B (emergence threshold)

In [ ]:
all_results = []

model = HookedTransformer.from_pretrained('pythia-2.8b', device=device)
print('--- Pythia 2.8B ---')
all_results.extend(run_prompts(model, 'pythia-2.8b', prompts))
del model
torch.mps.empty_cache() if device == 'mps' else None

### Pythia 1B (predicted dead zone)

In [ ]:
model = HookedTransformer.from_pretrained('pythia-1b', device=device)
print('--- Pythia 1B ---')
all_results.extend(run_prompts(model, 'pythia-1b', prompts))
del model
torch.mps.empty_cache() if device == 'mps' else None

### GPT-2 Large (cross-architecture check)

In [ ]:
model = HookedTransformer.from_pretrained('gpt2-large', device=device)
print('--- GPT-2 Large ---')
all_results.extend(run_prompts(model, 'gpt2-large', prompts))
del model
torch.mps.empty_cache() if device == 'mps' else None

## Results

In [ ]:
df = pd.DataFrame(all_results)
df.to_csv('../results/elicitation_robustness_raw.csv', index=False)
print(f'Saved {len(df)} results')
df.head()

In [ ]:
# View results by model and template type
for model_name in df['model'].unique():
    print(f'\n{"=" * 60}')
    print(f'{model_name}')
    print(f'{"=" * 60}')
    model_df = df[df['model'] == model_name]
    for _, row in model_df.iterrows():
        print(f"\n[{row['template_type']:16}] {row['concept']}")
        print(f"  Prompt:     {row['prompt']}")
        print(f"  Completion: {row['completion'][:120]}")